## Verbalising Thoughts from an episodic Knowledge Graph using an LLM

The LLM is called through ```ollama``. Ollama needs tbe installed locally. See [instructions](https://ollama.com).

In [9]:
import json
import pprint
from langchain_ollama import ChatOllama

In [ ]:
#! ollama pull llama3.2

In [2]:
model_name = "llama3.2"
ollama_client = ChatOllama(
            model= model_name,
            temperature=0.1,
            num_ctx=4096,   # limits KV-cache size, avoids GPU OOM
            # other params ...
        )


The ```data``` folder contains JSON files with basic responses generated from a Leolani episodic Knowledge Graph. In the next cells, we load such a response file and call the LLM to verbalise it.

In [10]:
path = "thoughts_for_Lunch_from_file_mybrain_with_events.trig.json"
file = open(path)
data = json.load(file)


In [11]:
print("Length of data", len(data))
for item in data:
    print(item, len(data[item]))


Length of data 5
untyped_entities 3
predicate_gaps 1
dangling_references 6
predicate_object_gaps 5
predicate_object_instances_gaps 4


## Instruct object type gap

In [37]:
UTTERANCE_TYPE = "question"
instruct_object_gap = {"role": "system", "content": f"You are an intelligent assistant. \
     I will give you as input: a triple with a subject, a predicate and a type of object.\
     You need to paraphrase the input in plain English as a {UTTERANCE_TYPE} for the object. \
     Use who for the type person, where for the type location, when for the type time and what for everything else. \
     Only reply with the short paraphrase of the input. \
     Do not give an explanation. \
     Do not explain what the subject and object is. \
     The response should be just the paraphrased text and nothing else."
           }

instruct_object_gap = {"role": "system", "content": f"You are an intelligent assistant. \
     You will receive as input a Python dictionary that expresses a gap in the knowledge about an instance of an event. \
     The dictionary has the following elements: subject, class, predicate, object type, peer_coverage, subject_triples, and object_triples.\
     The subject element represents the unique identifier for the instance of an event. \
     The class element represents what kind of event it is. \
     The peer_coverage element represents \
     The object type indicates the type of object that is missing in the Knowledge Graph. \
     The predicate represents the semantic role of the missing object. \
     The subject_triples give all the other knowledge that is known about the event instance. \
     The object_triples tell you more about the objects of the subsets. \
     First, you need to tell the user what is known about the event as given in the subject_triples.\
     Only use the information from subject_triples and the object_triples as known. \
     Next, you inform the user about the peer_coverage of instances of the same type that do have the knowledge that is missing for this event instance.\
     Finally, formulate a question to the user to confirm or deny if this is also the case for the current event.\
     Do not use the word instance in the paraphrase.\
     Do not use we but use I.\
     Do not use the words 'patient', 'subject', 'predicate', 'object' in the paraphrase. \
     Do not give an explanation. \
     Do not explain what the subject and object is. \
     The response should be just the paraphrased text and nothing else."
           }

In [38]:
for gap in data["predicate_object_gaps"]:
    pprint.pp(gap)
    if "subject" in gap and "predicate" in gap and "object_type" in gap:
#        triple_text = gap["subject"]+" "+gap["predicate"]+" "+gap["object_type"]
#        print(triple_text)
        prompt = [instruct_object_gap, {"role": "user", "content": json.dumps(gap)}]
        response = ollama_client.invoke(prompt)
        print(response.content)
        print()




{'class': 'http://cltl.nl/leolani/n2mu/Lunch',
 'predicate': 'http://cltl.nl/leolani/n2mu/hasPatient',
 'object_type': 'http://cltl.nl/leolani/n2mu/Cheese',
 'subject': 'http://cltl.nl/leolani/n2mu/lunch3',
 'missing_count': 2,
 'total_instances': 4,
 'peer_coverage': '2/4'}
Here's what I know about this event: 
The lunch has a type of cheese. 
Two out of four other lunches have the same type of cheese.

Does this also apply to this lunch?

{'class': 'http://cltl.nl/leolani/n2mu/Lunch',
 'predicate': 'http://cltl.nl/leolani/n2mu/hasPatient',
 'object_type': 'http://cltl.nl/leolani/n2mu/Cheese',
 'subject': 'http://cltl.nl/leolani/n2mu/lunch4',
 'missing_count': 2,
 'total_instances': 4,
 'peer_coverage': '2/4'}
Here's what I know about this event: 
The lunch has a type of cheese. 
Two out of four other lunches have the same type of cheese.

Do you think that's also true for this lunch?

{'class': 'http://cltl.nl/leolani/n2mu/Lunch',
 'predicate': 'http://cltl.nl/leolani/n2mu/hasPatient